# 双支天线长度预测 MLP 训练笔记
本 Notebook 展示了如何从包含反射系数幅值和相位的 CSV 数据集中训练一个预测双支天线最佳长度 (l1, l2) 的多层感知机。
Notebook 仅依赖 Python 标准库，方便在依赖受限的环境中直接运行。


## 如何在 Jupyter Notebook 中运行训练
1. 打开本 Notebook 后，先执行下方 "导入所需模块" 的代码单元，以加载 `simple_mlp` 辅助模块。
2. 接着运行 "配置区域" 单元，根据需要调整 CSV 路径或训练超参数。
3. 如果你还没有准备好的数据，可以继续运行 "生成演示用 CSV" 单元，它会在 `data/` 目录下生成 2000 条训练样本。
4. 最重要的训练步骤在 "定义并训练模型" 单元，运行该单元即可启动训练过程。
5. 最后运行 "训练集与测试集表现评估" 单元，查看 MSE 以及前几条预测结果。

你也可以通过菜单 "运行 -> 运行全部" 一次性顺序执行所有单元，Notebook 会自动在缺失数据时生成演示数据并完成训练。


In [ ]:
# 导入所需模块（全部来自标准库或 simple_mlp 辅助模块）
from pathlib import Path
import math
import random

from simple_mlp import (
    SimpleMLP,
    Standardizer,
    read_csv_dataset,
    mean_squared_error,
    format_predictions,
)


In [ ]:
# ===== 配置区域 =====
project_dir = Path('.')
train_csv_path = project_dir / 'data' / 'train.csv'
test_csv_path = project_dir / 'data' / 'test.csv'

feature_columns = ['reflection_magnitude_L', 'reflection_phase']
target_columns = ['optimal_length_l1', 'optimal_length_l2']

NUM_EPOCHS = 100
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
L1_COEFF = 1e-5
RANDOM_SEED = 42
random.seed(RANDOM_SEED)


In [ ]:
        # ===== 可选：生成演示用 CSV（当指定文件不存在时） =====
        def generate_demo_csv(path: Path, with_targets: bool = True, num_rows: int = 2000) -> None:
            path.parent.mkdir(parents=True, exist_ok=True)
            rng = random.Random(RANDOM_SEED)
            rows = []
            for _ in range(num_rows):
                magnitude = rng.uniform(0.0, 1.0)
                phase = rng.uniform(-math.pi, math.pi)
                row = {
                    'reflection_magnitude_L': f"{magnitude:.6f}",
                    'reflection_phase': f"{phase:.6f}",
                }
                if with_targets:
                    real_part = magnitude * math.cos(phase)
                    imag_part = magnitude * math.sin(phase)
                    base_length = 0.3 + 0.25 * magnitude
                    delta = 0.15 * real_part - 0.1 * imag_part
                    coupling = 0.05 * math.sin(2 * phase)
                    noise = 0.03 * rng.gauss(0, 1)
                    length_l1 = max(0.0, min(7.0, 7 * (base_length + delta + noise)))
                    length_l2 = max(0.0, min(7.0, 7 * (base_length - delta + coupling + noise)))
                    row['optimal_length_l1'] = f"{length_l1:.6f}"
                    row['optimal_length_l2'] = f"{length_l2:.6f}"
                rows.append(row)

            header = ['reflection_magnitude_L', 'reflection_phase']
            if with_targets:
                header.extend(['optimal_length_l1', 'optimal_length_l2'])

            with path.open('w', encoding='utf-8', newline='') as handle:
                handle.write(','.join(header) + '
')
                for row in rows:
                    handle.write(','.join(row.get(column, '') for column in header) + '
')
            print(f"Demo CSV generated at: {path} (rows={num_rows}, targets={'yes' if with_targets else 'no'})")

        if not train_csv_path.exists():
            generate_demo_csv(train_csv_path, with_targets=True, num_rows=2000)
        if not test_csv_path.exists():
            generate_demo_csv(test_csv_path, with_targets=True, num_rows=400)


In [ ]:
# ===== 读取数据并进行标准化 =====
train_features_raw, train_targets = read_csv_dataset(train_csv_path, feature_columns, target_columns)
try:
    test_features_raw, test_targets = read_csv_dataset(test_csv_path, feature_columns, target_columns)
except ValueError as exc:
    print(f"Test CSV 缺少目标列: {exc}. 将仅基于输入特征进行预测。")
    test_features_raw, _ = read_csv_dataset(test_csv_path, feature_columns, [])
    test_targets = None

standardizer = Standardizer.fit(train_features_raw)
train_features = standardizer.transform(train_features_raw)
test_features = standardizer.transform(test_features_raw)

print(f'Training samples: {len(train_features)}')
print(f'Test samples: {len(test_features)}')


In [ ]:
# ===== 定义并训练模型 =====
mlp = SimpleMLP(
    layer_sizes=[len(feature_columns), 32, 16, len(target_columns)],
    learning_rate=LEARNING_RATE,
    l1_coeff=L1_COEFF,
    seed=RANDOM_SEED,
)
loss_history = mlp.train(
    features=train_features,
    targets=train_targets,
    epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    verbose_interval=10,
)
print(f'最后一个 epoch 的平均批次损失: {loss_history[-1]:.6f}')


In [ ]:
        # ===== 训练集与测试集表现评估 =====
        train_predictions = mlp.predict_batch(train_features)
        train_mse = mean_squared_error(train_predictions, train_targets)
        print(f'Train MSE: {train_mse:.6f}')

        test_predictions = mlp.predict_batch(test_features)
        if test_targets is not None:
            test_mse = mean_squared_error(test_predictions, test_targets)
            print(f'Test MSE: {test_mse:.6f}')
        else:
            print('Test CSV 未提供目标列，无法计算 MSE。')

        print('
测试集预测示例:')
        print(format_predictions(test_features_raw, test_predictions, test_targets, feature_columns, target_columns))


## 小结
- 通过标准化反射系数输入特征并训练 `SimpleMLP`，可以在受限环境中完成双输出天线长度预测。
- 可以根据需要调整隐藏层规模、训练轮数以及正则化强度，以获得更好的效果。
